# TD3 re-run — ARCHIVED

**Status: superseded. Kept for the record; do not build on this.**

The TD3 deep-RL approach the project started with. It was dropped for a diagnosable reason:
the untrained actor only ever proposed amplitudes between roughly 1600 and 3500 µA, so it
never sampled the low-amplitude region where the good settings turned out to be. It trained
on half the action space and lost to fixed open-loop stimulation in 6/6 runs.

The T3P half of this notebook — the live work — is now `01_t3p_sweeps.ipynb`.

Needs `dbs_core.py`'s superseded half to run.

# TD3 for DBS — clean re-run

Modules at **2.6.0**. Earlier runs used an RMS value corrupted by integer overflow
(`dbs ** 2` wrapping negative), so the energy term was wrong throughout. All array
arithmetic is now done in float64.

Run in order. Do not skip cell 4.

| # | Step | MATLAB | Time |
|---|---|---|---|
| 1 | Version check | no | instant |
| 2 | Offline checks | no | ~1 min |
| 3 | Start MATLAB | yes | ~1 min |
| 4 | Live env checks | yes | ~5 min |
| 5 | Choose rms_max | no | instant |
| 6 | Apply rms_max | yes | ~5 min |
| 7 | Fixed-policy baselines | yes | ~10 min |
| 8 | Short run (30 eps) | yes | ~15 min |
| 9 | Full run | yes | ~3 hrs |
| 10 | Evaluate | yes | ~15 min |
| 11 | Switch to 32 and repeat | yes | ~3 hrs |
| 12 | Compare | no | instant |

## 1. Version check

In [ ]:
import importlib
import numpy as np
import dbs_core, dbs_plots, dbs_checks
for m in (dbs_core, dbs_plots, dbs_checks):
    importlib.reload(m)

from dataclasses import replace
from dbs_core import Config, MatlabEnv, MockEnv, TD3Agent, train, evaluate

assert dbs_checks.versions(), "stale file — re-download all three .py files, restart kernel"

## 2. Offline checks

Expect **13 passed, 0 failed** for each. Says nothing about MATLAB — that is cell 4.

In [ ]:
assert dbs_checks.offline(64)
print()
assert dbs_checks.offline(32)

## 3. Start MATLAB

Once per session. If you edit a `.py` file, restart the kernel and start again from
cell 1 — do not reload, it leaves objects bound to stale classes.

`rms_max` is a placeholder here; cell 5 determines the real value.

In [ ]:
cfg = Config(
    hidden_dim=64,
    pd_value=0.4,
    tmax=1500,
    steps_per_episode=15,
    n_episodes=500,
    checkpoint_every=50,
    rms_max=300.0,
    freq_max=185.0,
    run_dir="runs",
    tag="v3",
)

menv = MatlabEnv(cfg, mat_path="bgn_vars.mat")
print("MATLAB engine started")
print("dbs dtype check:")
menv.reset()
d = menv._load()
for k in ("dbs", "vgi", "sgis"):
    print(f"  {k:<6} dtype {str(d[k].dtype):<10} max {np.abs(d[k]).max():.1f}")

## 4. Live environment checks

Probes the real model at seven actions, 3 settle steps then 4 measured. All 11 checks
must pass. The RMS column is the one that changed — expect values well above the
20-40 range seen before the float64 fix.

In [ ]:
ok, samples, rows = dbs_checks.live(menv, cfg)
print()
print("observed rms range:", tuple(round(v, 1) for v in dbs_checks.rms_range(samples)))

## 5. Choose rms_max

Candidates are scaled from the observed RMS. The suggestion is the largest value whose
best action still uses the **lowest** amplitude — a larger margin is not better if it is
bought by rewarding maximum current.

In [ ]:
probes = {k: {"sgis": float(np.mean([r["sgis"] for r in v])),
              "rms": float(np.mean([r["rms"] for r in v])),
              "freq": k[0], "amp": k[1]}
          for k, v in samples.items()}
suggestion = dbs_checks.tune_rms_max(cfg, probes)

## 6. Apply it

Edit the number to whatever cell 5 suggested, then re-check.

In [ ]:
cfg = replace(cfg, rms_max=1200.0)
menv.cfg = cfg
print("rms_max =", cfg.rms_max)
ok, samples, rows = dbs_checks.live(menv, cfg, verbose=True)
assert ok

## 7. Fixed-policy baselines

Evaluates constant-parameter policies through the same episode structure the agent sees.
This is the o-DBS baseline — the number the learned policy has to beat, and a row in the
results table either way.

In [ ]:
# fixed = dbs_checks.baseline(menv, cfg, n_episodes=5)

In [ ]:
import json, time, traceback
from pathlib import Path

PD_VALUES = [0.3, 0.6, 1.0]
HIDDEN_DIMS = [64, 32]
N_EPISODES = 300

summary = {}
t_start = time.time()

for pd in PD_VALUES:
    for hidden in HIDDEN_DIMS:
        run_cfg = replace(cfg, pd_value=pd, hidden_dim=hidden, n_episodes=N_EPISODES)
        name = run_cfg.name()
        print(f"\n{'='*70}\n{name}  |  started {time.strftime('%H:%M:%S')}\n{'='*70}")
        entry = {"pd": pd, "hidden": hidden, "started": time.strftime("%Y-%m-%d %H:%M:%S")}
        try:
            menv.cfg = run_cfg
            ok, samples, _ = dbs_checks.live(menv, run_cfg, verbose=False)
            entry["env_checks_passed"] = bool(ok)
            if not ok:
                print(f"  !! env checks failed at pd={pd} — training anyway, treat with caution")

            t0 = time.time()
            agent, logger = train(run_cfg, menv, verbose=True)
            entry["train_minutes"] = round((time.time() - t0) / 60, 1)

            ev = evaluate(run_cfg, menv, agent, n_episodes=10)
            entry.update({
                "eval_reward": round(float(np.mean([r["reward"] for r in ev])), 4),
                "eval_freq": round(float(np.mean([r["freq"] for r in ev])), 1),
                "eval_amp": round(float(np.mean([r["amp"] for r in ev])), 0),
                "eval_sgis": round(float(np.mean([r["sgis"] for r in ev])), 0),
                "run_dir": str(logger.dir),
            })

            fixed = dbs_checks.baseline(menv, run_cfg, n_episodes=3)
            best = max((k for k in fixed if k[0] > 0), key=lambda k: fixed[k]["all"])
            entry["best_fixed"] = {"freq": best[0], "amp": best[1],
                                   "reward": round(fixed[best]["all"], 4),
                                   "sgis": round(fixed[best]["sgis"], 0)}

            dbs_plots.plot_training(logger.dir)
            entry["status"] = "ok"
            print(f"  done in {entry['train_minutes']} min | "
                  f"eval {entry['eval_reward']:+.4f} | sgis {entry['eval_sgis']:.0f}")
        except Exception as e:
            entry["status"] = "failed"
            entry["error"] = f"{type(e).__name__}: {e}"
            print(f"  !! FAILED: {type(e).__name__}: {e}")
            traceback.print_exc()

        summary[name] = entry
        Path("runs").mkdir(exist_ok=True)
        Path("runs/batch_summary.json").write_text(json.dumps(summary, indent=2))

print(f"\n{'='*70}")
print(f"batch finished in {(time.time()-t_start)/60:.1f} min")
print(f"{'name':<28} {'status':<8} {'eval':>9} {'sgis':>7} {'best fixed':>11}")
print("-" * 70)
for name, e in summary.items():
    bf = f"{e['best_fixed']['reward']:+.4f}" if "best_fixed" in e else "—"
    ev = f"{e['eval_reward']:+.4f}" if "eval_reward" in e else "—"
    sg = f"{e['eval_sgis']:.0f}" if "eval_sgis" in e else "—"
    print(f"{name:<28} {e['status']:<8} {ev:>9} {sg:>7} {bf:>11}")

## 8. Short run

30 episodes to confirm the loop survives.

In [ ]:
import json
s = json.load(open("runs/batch_summary.json"))
print(f"{'run':<26} {'ok':<6} {'envchk':<7} {'eval':>9} {'sgis':>7} {'freq':>7} {'amp':>7} {'best fixed':>11} {'bf sgis':>8}")
print("-" * 96)
for name, e in s.items():
    bf = e.get("best_fixed", {})
    print(f"{name:<26} {e['status']:<6} {str(e.get('env_checks_passed','?')):<7} "
          f"{e.get('eval_reward','—'):>9} {e.get('eval_sgis','—'):>7} "
          f"{e.get('eval_freq','—'):>7} {e.get('eval_amp','—'):>7} "
          f"{bf.get('reward','—'):>11} {bf.get('sgis','—'):>8}")

In [ ]:
smoke_cfg = replace(cfg, n_episodes=30, checkpoint_every=10, tag="v3_smoke")
smoke_env = MatlabEnv(smoke_cfg, mat_path="bgn_vars.mat")
smoke_agent, smoke_logger = train(smoke_cfg, smoke_env, verbose=True)
smoke_env.close()
dbs_plots.plot_training(smoke_logger.dir);

## 9. Full run

First ~33 episodes are warmup with no learning. Checkpoints every 50 to `runs/<name>/`.

In [ ]:
agent, logger = train(cfg, menv, verbose=True)
print("\nrun directory:", logger.dir)

## 10. Evaluate

In [ ]:
eval_rows = evaluate(cfg, menv, agent, n_episodes=15)
print(f"avg reward: {np.mean([r['reward'] for r in eval_rows]):+.4f}")
print(f"avg freq:   {np.mean([r['freq'] for r in eval_rows]):.1f} Hz")
print(f"avg amp:    {np.mean([r['amp'] for r in eval_rows]):.0f} uA/cm2")
print(f"avg sgis:   {np.mean([r['sgis'] for r in eval_rows]):.0f}")

best_fixed = max((k for k in fixed if k[0] > 0), key=lambda k: fixed[k]["all"])
print(f"\nbest fixed: {best_fixed[0]}Hz/{best_fixed[1]} -> {fixed[best_fixed]['all']:+.4f}, "
      f"sgis {fixed[best_fixed]['sgis']:.0f}")

dbs_plots.plot_training(logger.dir)
dbs_plots.plot_eval(logger.dir, eval_rows);

## 11. Switch to 32

Change `hidden_dim` only. Everything else — seed, rms_max, episodes — stays identical,
which is what makes this a controlled comparison.

In [ ]:
menv.close()
cfg = replace(cfg, hidden_dim=32)
menv = MatlabEnv(cfg, mat_path="bgn_vars.mat")
print(cfg.name(), "| rms_max", cfg.rms_max, "| episodes", cfg.n_episodes)

In [ ]:
agent32, logger32 = train(cfg, menv, verbose=True)
eval32 = evaluate(cfg, menv, agent32, n_episodes=15)
print(f"\navg reward: {np.mean([r['reward'] for r in eval32]):+.4f}")
print(f"avg sgis:   {np.mean([r['sgis'] for r in eval32]):.0f}")

## 12. Compare

In [ ]:
import csv, statistics as st

def summarize(d, last=150):
    rows = list(csv.DictReader(open(f"runs/{d}/metrics.csv")))
    g = lambda k: [float(x[k]) for x in rows][-last:]
    return g("base_reward"), g("avg_freq"), g("avg_amp"), g("avg_sgis")

names = ["td3_h32_pd0.4_v3", "td3_h64_pd0.4_v3"]
res = {}
for d in names:
    r, f, a, s = summarize(d)
    res[d] = r
    print(f"{d}: reward {st.mean(r):+.4f} +/- {st.stdev(r):.4f} | "
          f"freq {st.mean(f):.1f} | amp {st.mean(a):.0f} | sgis {st.mean(s):.0f}")

r32, r64 = res[names[0]], res[names[1]]
pooled = ((st.stdev(r32) ** 2 + st.stdev(r64) ** 2) / 2) ** 0.5
print(f"\ngap (64-32) {st.mean(r64) - st.mean(r32):+.4f} = "
      f"{abs(st.mean(r64) - st.mean(r32)) / pooled:.2f} sd")

dbs_plots.compare_runs([f"runs/{n}" for n in names],
                       save_to="runs/compare_32_vs_64_v3.png");

## Shut down

In [ ]:
menv.close()
print("engine closed")